<a href="https://colab.research.google.com/github/thbieltraining/C/blob/main/CALCULADORA_SENAI_ADME2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from datetime import datetime
from decimal import Decimal, ROUND_HALF_UP

SALARIO_MINIMO_2026 = 1621.00

def arredonda(valor):
    """Arredondamento padrão SENAI (Half Up a 2 casas decimais)"""
    d = Decimal(str(valor))
    return float(d.quantize(Decimal('0.01'), rounding=ROUND_HALF_UP))

TABELA_INSS = [
    (1621.00, 0.075, 0.00),
    (2902.84, 0.09, 24.31),
    (4304.87, 0.12, 111.40),
    (8475.55, 0.14, 197.50),
]

TABELA_IRRF = [
    (2428.80, 0.00, 0.00),
    (2826.65, 0.075, 182.16),
    (3751.05, 0.15, 394.16),
    (4664.68, 0.225, 675.49),
    (float('inf'), 0.275, 908.73),
]

NOMES_MESES = [
    "", "Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
    "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"
]

def calc_inss(base):
    for teto, aliq, ded in TABELA_INSS:
        if base <= teto:
            return arredonda(base * aliq - ded)
    return arredonda(8475.55 * 0.14 - 197.50)

def calc_irrf(base_irrf, bruto_total_tributavel):
    irrf_bruto = 0.0
    for teto, aliq, ded in TABELA_IRRF:
        if base_irrf <= teto:
            irrf_bruto = base_irrf * aliq - ded
            break

    if bruto_total_tributavel <= 5000:
        return 0.0
    else:
        return arredonda(max(0.0, irrf_bruto))

def ler_numero(mensagem, valor_padrao, tipo=float):
    entrada = input(f"{mensagem} [{valor_padrao}]: ").strip()
    if entrada == "":
        return tipo(valor_padrao)
    return tipo(entrada.replace(',', '.'))

def capturar_adicionais_risco(salario_base):
    tem_adicional = input("Possui adicional de Insalubridade ou Periculosidade? (s/n) [n]: ").strip().lower()
    if tem_adicional not in ['s', 'sim']:
        return 0.0, "Nenhum", 0.0

    tipo = input("Qual o adicional? (i = Insalubridade, p = Periculosidade) [p]: ").strip().lower()

    if tipo == 'p':
        valor_adicional = arredonda(salario_base * 0.30)
        return valor_adicional, "Periculosidade (30%)", 0.30

    elif tipo == 'i':
        print("\nGraus de Insalubridade:")
        print("1 = Mínimo (10%) | 2 = Médio (20%) | 3 = Máximo (40%)")
        grau = ler_numero("Informe o grau (1, 2 ou 3)", 2, int)

        percentuais = {1: 0.10, 2: 0.20, 3: 0.40}
        perc = percentuais.get(grau, 0.20)

        valor_adicional = arredonda(SALARIO_MINIMO_2026 * perc)
        return valor_adicional, f"Insalubridade ({perc*100:.0f}%)", perc

    return 0.0, "Nenhum", 0.0

def calcular_hora_base(salario_com_periculosidade, carga_horaria):
    return arredonda(salario_com_periculosidade / carga_horaria)

def calcular_he_duas_etapas(hora_base, multiplicador, qtd_horas):
    if qtd_horas <= 0:
        return 0.0, 0.0

    hora_unitaria_he = arredonda(hora_base * multiplicador)
    total_he = arredonda(hora_unitaria_he * qtd_horas)
    return hora_unitaria_he, total_he

def capturar_faltas_multiplos_meses(mes_vigente):
    """
    Captura faltas no padrão m-s-d aceitando múltiplos valores por vírgula.
    Ex: 4-3-1, 5-1-1, 9-2-2 (Mês 4 Sem 3 1d | Mês 5 Sem 1 1d | Mês 9 Sem 2 2d)
    Valida o limite de 14 faltas de forma independente por mês.
    """
    houve_faltas = input("Houve faltas injustificáveis? (s/n) [n]: ").strip().lower()
    if houve_faltas not in ['s', 'sim']:
        return 0.0, 0, {}, 0.0, 0

    print("\n[FALTAS FORMATO m-s-d MÚLTIPLOS]")
    print("Informe pares separados por vírgula. Ex: '4-3-1, 5-1-1, 9-2-2'")
    entrada = input(f"Informe as faltas [ex: {mes_vigente}-1-1]: ").strip() or f"{mes_vigente}-1-1"

    padrao = r'(\d+)\s*-\s*([1-4])\s*-\s*(\d+)'
    ocorrencias = re.findall(padrao, entrada)

    # Estrutura para agrupar por mês
    faltas_por_mes = {}

    for mes, semana, dias in ocorrencias:
        m, s, d = int(mes), int(semana), int(dias)

        if m not in faltas_por_mes:
            faltas_por_mes[m] = {1: 0, 2: 0, 3: 0, 4: 0, 'total_dias': 0}

        dias_atuais_mes = faltas_por_mes[m]['total_dias']
        if dias_atuais_mes + d > 14:
            print(f"⚠️ Atenção (Mês {m}): Limite máximo de 14 faltas atingido! Ajustando {d} dias para {14 - dias_atuais_mes} dias.")
            d = 14 - dias_atuais_mes

        if d > 0:
            faltas_por_mes[m][s] += d
            faltas_por_mes[m]['total_dias'] += d

    # Totais Globais
    total_dias_global = sum(dados['total_dias'] for dados in faltas_por_mes.values())
    total_semanas_dsr_global = sum(
        sum(1 for s in range(1, 5) if dados[s] > 0)
        for dados in faltas_por_mes.values()
    )

    # Totais Exclusivos do Mês Vigente (para abatimento na Folha Atual)
    dias_mes_vigente = faltas_por_mes.get(mes_vigente, {}).get('total_dias', 0)
    semanas_dsr_vigente = sum(1 for s in range(1, 5) if faltas_por_mes.get(mes_vigente, {}).get(s, 0) > 0)

    return float(total_dias_global), total_semanas_dsr_global, faltas_por_mes, float(dias_mes_vigente), semanas_dsr_vigente

def calcular_desconto_faltas(salario_base, valor_adicional, total_dias_faltados, semanas_perda_dsr):
    if total_dias_faltados <= 0:
        return 0.0, 0.0, 0.0

    remuneracao_total = salario_base + valor_adicional
    valor_dia = arredonda(remuneracao_total / 30)

    v_faltas = arredonda(valor_dia * total_dias_faltados)
    v_dsr_falta = arredonda(valor_dia * semanas_perda_dsr)
    total_desconto = arredonda(v_faltas + v_dsr_falta)

    return v_faltas, v_dsr_falta, total_desconto

def capturar_e_calcular_ferias(salario_remuneracao, mes_competencia, ano_competencia=2026):
    quer_ferias = input("Deseja programar/calcular Férias? (s/n) [n]: ").strip().lower()
    if quer_ferias not in ['s', 'sim']:
        return None

    entrada_periodo = input("Quando as férias serão tiradas? ('mesmo' ou ex: '1-2026') [mesmo]: ").strip().lower() or "mesmo"

    m_gozo, a_gozo = mes_competencia, ano_competencia
    if entrada_periodo != "mesmo":
        match = re.match(r'(\d+)-(\d{4})', entrada_periodo)
        if match:
            m_gozo = int(match.group(1))
            a_gozo = int(match.group(2))

    eh_mes_vigente = (m_gozo == mes_competencia and a_gozo == ano_competencia)
    dias_gozo = ler_numero("Dias de Férias gozados (ex: 30, 20, 15)", 30, int)
    vender_dez_dias = input("Houve abono pecuniário (vendeu 10 dias)? (s/n) [n]: ").strip().lower() in ['s', 'sim']

    valor_dia = arredonda(salario_remuneracao / 30)

    val_ferias_gozo = arredonda(valor_dia * dias_gozo)
    terco_ferias = arredonda(val_ferias_gozo / 3)
    total_tributavel = arredonda(val_ferias_gozo + terco_ferias)

    val_abono, terco_abono, total_isento = 0.0, 0.0, 0.0
    if vender_dez_dias:
        val_abono = arredonda(valor_dia * 10)
        terco_abono = arredonda(val_abono / 3)
        total_isento = arredonda(val_abono + terco_abono)

    total_bruto_ferias = arredonda(total_tributavel + total_isento)

    return {
        'mes_gozo': m_gozo,
        'ano_gozo': a_gozo,
        'eh_mes_vigente': eh_mes_vigente,
        'dias_gozo': dias_gozo,
        'val_ferias': val_ferias_gozo,
        'terco_ferias': terco_ferias,
        'total_tributavel': total_tributavel,
        'vendeu_abono': vender_dez_dias,
        'val_abono': val_abono,
        'terco_abono': terco_abono,
        'total_isento': total_isento,
        'total_bruto': total_bruto_ferias
    }

def capturar_e_calcular_13(salario_remuneracao, dependentes, mes_competencia, ano_competencia=2026):
    quer_13 = input("Deseja calcular 13º Salário? (s/n) [n]: ").strip().lower()
    if quer_13 not in ['s', 'sim']:
        return None

    entrada_periodo = input("Em qual mês/ano o 13º será pago? [12-2026]: ").strip().lower() or "12-2026"

    m_pago, a_pago = 12, ano_competencia
    if entrada_periodo != "mesmo":
        match = re.match(r'(\d+)-(\d{4})', entrada_periodo)
        if match:
            m_pago = int(match.group(1))
            a_pago = int(match.group(2))
    else:
        m_pago = mes_competencia

    eh_mes_vigente = (m_pago == mes_competencia and a_pago == ano_competencia)
    opcao = input("Qual a modalidade? [1=1ª Parcela, 2=2ª Parcela, d=Duas Parcelas] [2]: ").strip().lower() or "2"
    avos = ler_numero("Número de avos/meses trabalhados no ano (1 a 12)", 12, int)

    val_13_bruto_total = arredonda((salario_remuneracao / 12) * avos)
    p1_bruta = arredonda(val_13_bruto_total * 0.5)

    inss_13 = calc_inss(val_13_bruto_total)
    base_irrf_13 = arredonda(val_13_bruto_total - inss_13 - (dependentes * 189.59))
    irrf_13 = calc_irrf(base_irrf_13, val_13_bruto_total)

    p2_bruta = arredonda(val_13_bruto_total - p1_bruta)
    p2_liquida = arredonda(p2_bruta - inss_13 - irrf_13)

    return {
        'mes_pago': m_pago,
        'ano_pago': a_pago,
        'eh_mes_vigente': eh_mes_vigente,
        'opcao': opcao,
        'avos': avos,
        'val_13_bruto_total': val_13_bruto_total,
        'p1_bruta': p1_bruta,
        'p2_bruta': p2_bruta,
        'inss_13': inss_13,
        'base_irrf_13': base_irrf_13,
        'irrf_13': irrf_13,
        'p2_liquida': p2_liquida
    }

def executar_calculadora():
    print("=" * 65)
    print("  CALCULADORA FOPAG SENAI 2026 - FALTAS MÚLTIPLAS M-S-D")
    print("=" * 65)

    # 1. Salário Fixo e Adicionais em 2 Etapas
    salario = ler_numero("Digite o Salário Fixo (R$)", 2950.00)
    val_adicional, desc_adicional, perc_adicional = capturar_adicionais_risco(salario)

    carga_horaria = ler_numero("Carga Horária Mensal em Horas (ex: 220, 200)", 220.0)

    # 2. Competência e Faltas Múltiplas (m-s-d)
    mes_competencia = ler_numero("Mês Vigente da Folha (1 a 12)", 9, int)
    ano_competencia = 2026

    print("-" * 65)
    d_global, s_dsr_global, mapa_faltas, d_vigente, s_dsr_vigente = capturar_faltas_multiplos_meses(mes_competencia)
    print("-" * 65)

    # 3. Horas Extras e Dependentes
    he50_qtd = ler_numero("Horas Extras 50% (quantidade)", 18.0)
    he70_qtd = ler_numero("Horas Extras 70% (quantidade)", 0.0)
    he100_qtd = ler_numero("Horas Extras 100% (quantidade)", 10.0)
    dependentes = ler_numero("Número de Dependentes", 1, int)

    # Remuneração base
    salario_para_he = salario + val_adicional if "Periculosidade" in desc_adicional else salario
    remuneracao_base_total = salario + val_adicional

    # 4. FÉRIAS E 13º SALÁRIO
    print("-" * 65)
    dados_ferias = capturar_e_calcular_ferias(remuneracao_base_total, mes_competencia, ano_competencia)
    print("-" * 65)
    dados_13 = capturar_e_calcular_13(remuneracao_base_total, dependentes, mes_competencia, ano_competencia)
    print("-" * 65)

    # 5. Outros Dados (VT)
    valor_vt_total = ler_numero("Valor TOTAL do VT gasto no mês (R$)", 180.00)

    # CÁLCULOS DA FOLHA (Calcula desconto do mês vigente)
    hora_base = calcular_hora_base(salario_para_he, carga_horaria)
    v_faltas, v_dsr_falta, total_faltas = calcular_desconto_faltas(salario, val_adicional, d_vigente, s_dsr_vigente)

    u_he50, v_he50 = calcular_he_duas_etapas(hora_base, 1.5, he50_qtd)
    u_he70, v_he70 = calcular_he_duas_etapas(hora_base, 1.7, he70_qtd)
    u_he100, v_he100 = calcular_he_duas_etapas(hora_base, 2.0, he100_qtd)
    total_he = arredonda(v_he50 + v_he70 + v_he100)

    # COMPOSIÇÃO DOS VALORES
    adicionais_bruto_total = total_he + val_adicional
    adicionais_tributaveis = total_he + val_adicional

    if dados_ferias and dados_ferias['eh_mes_vigente']:
        adicionais_bruto_total += dados_ferias['total_bruto']
        adicionais_tributaveis += dados_ferias['total_tributavel']

    bruto_mensal_total = arredonda(salario - total_faltas + adicionais_bruto_total)
    base_tributavel_mensal = arredonda(salario - total_faltas + adicionais_tributaveis)

    inss_mensal = calc_inss(base_tributavel_mensal)
    base_irrf_mensal = arredonda(base_tributavel_mensal - inss_mensal - (dependentes * 189.59))
    irrf_mensal = calc_irrf(base_irrf_mensal, base_tributavel_mensal)

    vt = arredonda(min(salario * 0.06, valor_vt_total))
    liquido_mensal = arredonda(bruto_mensal_total - inss_mensal - irrf_mensal - vt)

    # IMPRESSÃO DETALHADA
    print("\n" + "=" * 65)
    print(f"    HOLERITE MENSAL - COMPETÊNCIA: {NOMES_MESES[mes_competencia].upper()}/{ano_competencia}")
    print("=" * 65)
    print(f"Salário Base        : R$ {salario:.2f}")
    if val_adicional > 0:
        print(f"Adicional Risco     : R$ {val_adicional:.2f} ({desc_adicional})")
    print(f"Hora Base ({carga_horaria:.0f}h)   : R$ {hora_base:.2f}")
    print("-" * 65)

    if d_global > 0:
        print("Resumo de Faltas Informadas (mês -> sem:dias):")
        for m, dados in mapa_faltas.items():
            print(f"  -> Mês {m:02d}: Sem1:{dados[1]}d | Sem2:{dados[2]}d | Sem3:{dados[3]}d | Sem4:{dados[4]}d (Total: {dados['total_dias']}d)")
        print(f"Desconto Faltas Mês Vigente ({d_vigente:.0f}d): - R$ {v_faltas:.2f}")
        print(f"Desconto DSR Mês Vigente ({s_dsr_vigente} sem)  : - R$ {v_dsr_falta:.2f}")
        print(f"Total Desc. Faltas Folha Atual : - R$ {total_faltas:.2f}")
        print("-" * 65)

    print(f"HE 50%  (Unit: R$ {u_he50:.2f}) x {he50_qtd}h  = R$ {v_he50:.2f}")
    print(f"HE 70%  (Unit: R$ {u_he70:.2f}) x {he70_qtd}h   = R$ {v_he70:.2f}")
    print(f"HE 100% (Unit: R$ {u_he100:.2f}) x {he100_qtd}h = R$ {v_he100:.2f}")
    print(f"Total Horas Extras  : R$ {total_he:.2f}")
    print("-" * 65)

    if dados_ferias:
        status_fer = "INCLUÍDO NA FOLHA" if dados_ferias['eh_mes_vigente'] else f"PROGRAMADO PARA {dados_ferias['mes_gozo']}/{dados_ferias['ano_gozo']}"
        print(f"FÉRIAS ({status_fer}):")
        print(f"  -> Gozo ({dados_ferias['dias_gozo']}d) + 1/3 (Tributável) : R$ {dados_ferias['total_tributavel']:.2f}")
        if dados_ferias['vendeu_abono']:
            print(f"  -> Abono (10d) + 1/3 (Isento/Ñ Trib.): R$ {dados_ferias['total_isento']:.2f}")
        print(f"  -> Bruto Total Férias             : R$ {dados_ferias['total_bruto']:.2f}")
        print("-" * 65)

    if dados_13:
        status_13 = "FOLHA SEPARADA DE 13º" if dados_13['eh_mes_vigente'] else f"PROGRAMADO PARA {dados_13['mes_pago']}/{dados_13['ano_pago']}"
        print(f"DEMONSTRATIVO DE 13º SALÁRIO ({status_13}):")
        print(f"  -> Total Bruto ({dados_13['avos']}/12 avos) : R$ {dados_13['val_13_bruto_total']:.2f}")
        print(f"  -> 1ª Parcela (50% Bruto)    : R$ {dados_13['p1_bruta']:.2f} (Sem Impostos)")
        print(f"  -> 2ª Parcela Bruta (50%)    : R$ {dados_13['p2_bruta']:.2f}")
        print(f"  -> Desconto INSS 13º         : - R$ {dados_13['inss_13']:.2f}")
        print(f"  -> Desconto IRRF 13º         : - R$ {dados_13['irrf_13']:.2f}")
        print(f"  -> 2ª Parcela LÍQUIDA        : R$ {dados_13['p2_liquida']:.2f}")
        print("-" * 65)

    print(f"SALÁRIO BRUTO TOTAL MÊS : R$ {bruto_mensal_total:.2f}")
    print(f"BASE DE CÁLCULO INSS/IRRF: R$ {base_tributavel_mensal:.2f}")
    print("-" * 65)
    print(f"Desconto INSS Mensal    : R$ {inss_mensal:.2f}")
    print(f"Base de IRRF Mensal     : R$ {base_irrf_mensal:.2f}")
    print(f"Desconto IRRF Mensal    : R$ {irrf_mensal:.2f}")
    print(f"Desconto VT (máx 6%)    : R$ {vt:.2f}")
    print("-" * 65)
    print(f"SALÁRIO LÍQUIDO MÊS     : R$ {liquido_mensal:.2f}")
    print("=" * 65)

if __name__ == "__main__":
    executar_calculadora()

  CALCULADORA FOPAG SENAI 2026 - FALTAS MÚLTIPLAS M-S-D
Digite o Salário Fixo (R$) [2950.0]: 2450
Possui adicional de Insalubridade ou Periculosidade? (s/n) [n]: s
Qual o adicional? (i = Insalubridade, p = Periculosidade) [p]: p
Carga Horária Mensal em Horas (ex: 220, 200) [220.0]: 200
Mês Vigente da Folha (1 a 12) [9]: 11
-----------------------------------------------------------------
Houve faltas injustificáveis? (s/n) [n]: s

[FALTAS FORMATO m-s-d MÚLTIPLOS]
Informe pares separados por vírgula. Ex: '4-3-1, 5-1-1, 9-2-2'
Informe as faltas [ex: 11-1-1]: 11-2-1,11-4-1
-----------------------------------------------------------------
Horas Extras 50% (quantidade) [18.0]: 0
Horas Extras 70% (quantidade) [0.0]: 13
Horas Extras 100% (quantidade) [10.0]: 17
Número de Dependentes [1]: 1
-----------------------------------------------------------------
Deseja programar/calcular Férias? (s/n) [n]: s
Quando as férias serão tiradas? ('mesmo' ou ex: '1-2026') [mesmo]: 1-2027
Dias de Férias goza